# Week 4 Presentation Brief — Variation A
## 🌾 Pesticide Degradation: Optimizing Spray Timing
**SCIE1500 — Semester 2, 2026 | Group 5 — presenting in Week 5**

> Work through all parts during the Week 4 lab. Your **10-minute Week 5 presentation** should cover: the problem, your model, key results, and agronomic implications.

---
## 📋 Scenario

A wheat farm in the Wheatbelt uses a pesticide that degrades over time:

$$C(t) = 120\,e^{-0.15t}$$

where $C$ is concentration (mg/L) and $t$ is days after application.

**Agronomic constraints:**
- Minimum effective: 30 mg/L (below this, pest control fails)
- Maximum safe: 150 mg/L (above, phytotoxicity)
- Aphids most vulnerable: days 5–10 after wheat head emergence

---
## 🎯 Your Task

| Part | Topic | Time |
|------|-------|------|
| A | Instantaneous rate of change $C'(t)$ | ~20 min |
| B | Effective protection window | ~20 min |
| C | Re-application strategy | ~15 min |
| D | Half-life verification | ~10 min |

In [ ]:
# Run first — loads libraries for this session
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Pesticide parameters
C0 = 120    # initial concentration (mg/L)
k  = 0.15   # degradation rate constant (per day)

def C(t):
    'Pesticide concentration (mg/L), t days after application.'
    return C0 * np.exp(-k * t)

def dC(t):
    'Rate of change of concentration (mg/L/day).'
    return -k * C0 * np.exp(-k * t)

print("Initial concentration C(0) =", C(0), "mg/L")
print("At t=5:", f"{C(5):.2f}", "mg/L")
print("At t=10:", f"{C(10):.2f}", "mg/L")

---
## Part A: Instantaneous Rate of Change (~20 min)

The rule $(e^{kt})' = k e^{kt}$ gives:
$$C'(t) = -k \cdot 120\,e^{-0.15t} = -0.15 \cdot C(t)$$

The rate of decrease is **proportional to the current concentration** — fast when concentrated, slower as it degrades.

In [ ]:
# A.1 — Concentration and rate at key days
print(f"{'Day':>5} | {'C(t) mg/L':>10} | {'C\'(t) mg/L/day':>16} | {'% per day':>9}")
print("-" * 50)
for t in [0, 5, 10, 15, 20]:
    pct = abs(dC(t) / C(t) * 100)
    print(f"{t:>5} | {C(t):>10.2f} | {dC(t):>16.2f} | {pct:>9.1f}%")

✏️ **Activity A:**
1. Evaluate $C'(5)$ and interpret in words including units.
2. Why is the percentage rate of decrease constant at 15%/day?
3. On day 10, is concentration decreasing faster or slower than on day 1?

```
Answers:
1. ...
2. ...
3. ...
```

---
## Part B: Effective Protection Window (~20 min)

In [ ]:
# B.1 — When does concentration drop below effective threshold?
C_min = 30    # minimum effective (mg/L)

# Solve: 120 * exp(-0.15 * t) = 30  →  t = -ln(30/120) / 0.15
t_end = -np.log(C_min / C0) / k
print(f"Protection ends at t = {t_end:.2f} days  ({t_end * 24:.0f} hours)")
print(f"Protection window: {t_end:.1f} days from application")

# If protection needed for 20 days, what initial concentration is required?
t_need = 20
C0_need = C_min * np.exp(k * t_need)
print(f"\nFor 20-day coverage, initial concentration needed: {C0_need:.1f} mg/L")
print(f"Safe? (< 150 mg/L limit): {C0_need < 150}")

In [ ]:
# B.2 — Plot concentration with protection window
t_vals = np.linspace(0, 30, 300)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 9))

ax1.plot(t_vals, C(t_vals), "steelblue", lw=2, label="Concentration C(t)")
ax1.axhline(C_min, color="red", ls="--", lw=1.5, label=f"Min effective ({C_min} mg/L)")
ax1.axhline(150,   color="orange", ls="--", lw=1.5, label="Max safe (150 mg/L)")
ax1.fill_between(t_vals, C_min, np.minimum(C(t_vals), 150),
                 where=(C(t_vals) >= C_min), alpha=0.2, color="green", label="Effective zone")
ax1.axvline(t_end, color="red", ls=":", label=f"Protection ends at {t_end:.1f} days")
ax1.set_ylabel("Concentration (mg/L)")
ax1.set_title("Pesticide Degradation")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2.plot(t_vals, dC(t_vals), "purple", lw=2, label="C'(t) = dC/dt")
ax2.axhline(0, color="k", lw=0.8)
ax2.set_xlabel("Days after application")
ax2.set_ylabel("Rate of change (mg/L/day)")
ax2.set_title("Rate of Pesticide Degradation")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part C: Re-Application Strategy (~15 min)

In [ ]:
# C.1 — Re-apply when concentration drops to 40 mg/L
C_reapply = 40

t_reapply = -np.log(C_reapply / C0) / k
print(f"Re-apply when C = {C_reapply} mg/L:  at t = {t_reapply:.1f} days")

# For 30-day coverage
n_applications = np.ceil(30 / t_reapply)
print(f"Days between applications: {t_reapply:.1f} days")
print(f"Total applications for 30-day coverage: {n_applications:.0f}")

# Total pesticide used
dose_per_ha = C0 * 500 / 1e6   # 120 mg/L × 500 L/ha → kg/ha
total_use = n_applications * dose_per_ha
print(f"\nDose per application: {dose_per_ha:.4f} kg/ha")
print(f"Total pesticide over 30 days: {total_use:.4f} kg/ha")

---
## Part D: Half-Life Verification (~10 min)

In [ ]:
# D.1 — Half-life: time for concentration to halve
t_half = np.log(2) / k
t_half_direct = -np.log(0.5) / k    # same formula

print(f"Half-life = ln(2)/k = ln(2)/{k} = {t_half:.2f} days")
print(f"Verification: C(0)/2 = {C0/2:.1f} mg/L")
print(f"C(t_half) = {C(t_half):.2f} mg/L  ✓" if abs(C(t_half) - C0/2) < 0.01 else "Check calculation")

# After two half-lives
print(f"\nAfter 2 half-lives ({2*t_half:.1f} days): {C(2*t_half):.2f} mg/L = {C(2*t_half)/C0*100:.1f}% of original")

---
## ✅ Presentation Checklist (Week 5, 10 minutes)

1. **Problem** (~2 min): What is the spray timing challenge?
2. **Model** (~3 min): Show $C(t)$ equation; explain what $k = 0.15$ means.
3. **Results** (~3 min): Protection window, re-application schedule — show the two-panel plot.
4. **Implications** (~2 min): What does this mean for aphid control in the critical growth window?